In [29]:
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path
import os
from dotenv import load_dotenv

# Load environment
BASE_DIR = Path.cwd().parent.parent.parent
ENV_PATH = BASE_DIR / '.env'
load_dotenv(dotenv_path=str(ENV_PATH))

# Database credentials
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

# SQLite database path
SQLITE_DB = BASE_DIR / 'data_engineer' / 'database' / 'loss_profit_renew.db'

print("=" * 80)
print("LOADING DATA FROM SQLITE TO POSTGRESQL")
print("=" * 80)
print()

# Step 1: Load from SQLite
print("STEP 1: Loading from SQLite")
print("─" * 80)
print(f"📁 Source: {SQLITE_DB}")

sqlite_engine = create_engine(f'sqlite:///{str(SQLITE_DB)}')
with sqlite_engine.connect() as conn:
    df = pd.read_sql_table('loss_profit', conn)

print(f"✅ Loaded {len(df):,} records from SQLite")
print(f"📋 Columns: {', '.join(df.columns)}")
print(f"✅ image_path column present: {'image_path' in df.columns}\n")

if 'image_path' not in df.columns:
    print("❌ ERROR: image_path column not found in SQLite database!")
    print("Please run: python pipelines/data_engineer/etl/transform/add_image_paths_final.py")
    exit(1)

# Step 2: Connect to PostgreSQL
print("STEP 2: Connecting to PostgreSQL")
print("─" * 80)
print(f"📊 Host: {DB_HOST}:{DB_PORT}")
print(f"📊 Database: {DB_NAME}")
print(f"📊 User: {DB_USER}\n")

postgres_connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
postgres_engine = create_engine(postgres_connection_string)

# Test connection
try:
    with postgres_engine.begin() as conn:
        conn.execute(text("SELECT 1"))
    print("✅ PostgreSQL connection successful\n")
except Exception as e:
    print(f"❌ PostgreSQL connection failed: {e}")
    exit(1)

# Step 3: Create table schema in PostgreSQL
print("STEP 3: Creating PostgreSQL table")
print("─" * 80)

with postgres_engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS fashion_recommendation CASCADE;"))
    conn.execute(text("""
        CREATE TABLE fashion_recommendation (
            id SERIAL PRIMARY KEY,
            item_id TEXT UNIQUE NOT NULL,
            purchase_count INTEGER NOT NULL DEFAULT 0,
            view_count INTEGER NOT NULL DEFAULT 0,
            price BIGINT NOT NULL,
            stocks INTEGER NOT NULL DEFAULT 0,
            sales BIGINT NOT NULL,
            stock_value_retail BIGINT NOT NULL,
            profit_status TEXT NOT NULL,
            conversion_rate NUMERIC(8, 2) NOT NULL DEFAULT 0.00,
            image_path VARCHAR(500) NOT NULL
        );
    """))
    conn.execute(text("CREATE INDEX idx_fashion_recommendation_item_id ON fashion_recommendation(item_id);"))
    conn.execute(text("CREATE INDEX idx_fashion_recommendation_profit_status ON fashion_recommendation(profit_status);"))
    conn.execute(text("CREATE INDEX idx_fashion_recommendation_image_path ON fashion_recommendation(image_path);"))

print("✅ Table created in PostgreSQL\n")

# Step 4: Insert data into PostgreSQL
print("STEP 4: Inserting data into PostgreSQL")
print("─" * 80)
print(f"Inserting {len(df):,} records...")

# Ensure data types are correct
df['price'] = df['price'].astype('int64')
df['stocks'] = df['stocks'].astype('int32')
df['purchase_count'] = df['purchase_count'].astype('int32')
df['view_count'] = df['view_count'].astype('int32')
df['sales'] = df['sales'].astype('int64')
df['stock_value_retail'] = df['stock_value_retail'].astype('int64')
df['conversion_rate'] = df['conversion_rate'].astype('float64')

# Insert using SQLAlchemy
df.to_sql(
    'fashion_recommendation',
    postgres_engine,
    if_exists='append',
    index=False,
    method='multi',
    chunksize=1000
)

print(f"✅ Successfully inserted {len(df):,} records into PostgreSQL\n")

LOADING DATA FROM SQLITE TO POSTGRESQL

STEP 1: Loading from SQLite
────────────────────────────────────────────────────────────────────────────────
📁 Source: /Users/miftahhadiyannoor/Documents/Fashion_Recommendation_Engineer/data_engineer/database/loss_profit_renew.db
✅ Loaded 300,000 records from SQLite
📋 Columns: item_id, purchase_count, view_count, price, stocks, sales, stock_value_retail, profit_status, conversion_rate, image_path
✅ image_path column present: True

STEP 2: Connecting to PostgreSQL
────────────────────────────────────────────────────────────────────────────────
📊 Host: localhost:5432
📊 Database: airflow
📊 User: airflow

✅ PostgreSQL connection successful

STEP 3: Creating PostgreSQL table
────────────────────────────────────────────────────────────────────────────────
✅ Table created in PostgreSQL

STEP 4: Inserting data into PostgreSQL
────────────────────────────────────────────────────────────────────────────────
Inserting 300,000 records...
✅ Successfully inser